#### T13 Network WS Resilient Typed Destination

Validate **`network_ws`** layer + destination wiring; preset keeps **simulated** transport so notebooks run without DNS (WSL/CI).

`examples/config/network_ws_basic.yaml`

**Expect:** Console + `t13_network_ws_layer.jsonl`; preset uses **simulated** `network_ws` (no live `wss://`). Set `use_real_websocket_transport: true` in YAML for a real echo when DNS works. <small>More: `examples/tutorials/notebooks/README.md`.</small>


**§0 — pip**  
<small>Optional if `import hydra_logger` already works.</small>


In [ ]:
%pip install -q "hydra-logger"
# Restart kernel if pip upgraded deps.


**§1 — Setup**  
<small>Notebook plumbing (usually **collapsed**): ``importlib`` loads ``jupyter_workspace.py``, then **one call** — ``prime_notebook_workspace()`` — (``shared.path_bootstrap`` → ``utility``). Clone root follows the loaded file, not Jupyter's cwd. Presets: ``examples/config/``. Details: `notebooks/README.md`.</small>


In [ ]:
import importlib.util
import os
import tempfile
from pathlib import Path

def _resolved_notebook_cwd() -> Path:
    try:
        return Path.cwd().resolve()
    except FileNotFoundError:
        fb = Path(tempfile.gettempdir()).resolve()
        os.chdir(fb)
        return fb

def _load_jupyter_workspace_module():
    _starts: list[Path] = []
    _env = os.environ.get("HYDRA_LOGGER_REPO", "").strip()
    if _env:
        _starts.append(Path(_env).expanduser().resolve())
    _starts.append(_resolved_notebook_cwd())
    _seen: set[str] = set()
    for _s in _starts:
        for _b in (_s, *_s.parents):
            _jw = _b / "examples" / "tutorials" / "jupyter_workspace.py"
            try:
                _key = str(_jw.resolve())
            except (OSError, RuntimeError):
                continue
            if _key in _seen:
                continue
            _seen.add(_key)
            if _jw.is_file():
                _spec = importlib.util.spec_from_file_location("_hydra_tutorial_jw", _jw)
                assert _spec and _spec.loader
                _mod = importlib.util.module_from_spec(_spec)
                _spec.loader.exec_module(_mod)
                return _mod
    raise FileNotFoundError(
        "Could not find examples/tutorials/jupyter_workspace.py. Clone hydra-logger, set "
        "HYDRA_LOGGER_REPO to the clone (or any directory inside it), or start Jupyter with cwd "
        "inside that clone."
    )

repo_root = _load_jupyter_workspace_module().prime_notebook_workspace()


**§2 — Imports**

In [ ]:
from hydra_logger import create_sync_logger


**§3 — Config path**  
<small>`CONFIG_PATH` is ``repo_root / "examples" / "config" / …`` — the same files you edit in the repository.</small>


In [ ]:
CONFIG_PATH = repo_root / "examples" / "config" / "network_ws_basic.yaml"


**§5 — Scenario**  
<small>Your hydra-logger usage pattern.</small>


In [ ]:
with create_sync_logger(
    config_path=CONFIG_PATH,
    strict_unknown_fields=True,
    name="tutorial-t13",
) as logger:
    logger.info("stream", layer="stream", context={"session_id": "s1"})


**§6 — Results**  
<small>Console + `t13_network_ws_layer.jsonl`; preset uses **simulated** `network_ws` (no live `wss://`). Set `use_real_websocket_transport: true` in YAML for a real echo when DNS works.</small>


**Iterate:** edit YAML under `examples/config/`, re-run **§5** (and **§6** if present). <small>Details: `notebooks/README.md`.</small>
